# Question 1 — Interactive waste-bin map

**Aditi Saha · HS25M101**

Interactive Folium web map of the waste-bin observations collected with QField
at IIT Tirupati, showing the locations of the recorded waste bins.
Each point shows a tooltip, a popup with the collector name,
and a photograph when one is available. Points are coloured by segregation status.

Photographs are stored in `web/photos` using the QField file names, so they
remain visible after the map is hosted.


In [1]:
# 1. Import libraries
from pathlib import Path

import folium
import geopandas as gpd
from branca.element import Element


In [2]:
# 2. Load the waste-bin GeoPackage
data_path = r"C:\Users\Ā\Documents\python_class\HS25M101\HS25M101.gpkg"

dustbins = gpd.read_file(
    data_path,
    layer="notes_point"
)

attachments = gpd.read_file(
    data_path,
    layer="notes_attachments"
)


In [3]:
# 3. Check CRS and convert to EPSG:4326
print(dustbins.crs)

dustbins = dustbins.to_crs(epsg=4326)
dustbins["geometry"] = dustbins.geometry.force_2d()

dustbins["timestamp"] = dustbins["timestamp"].astype(str)
dustbins["title"] = dustbins["title"].fillna("No title")
dustbins["note"] = dustbins["note"].fillna("No note")
dustbins["collector"] = "Aditi Saha"


EPSG:4326


In [4]:
# 4. Join photograph names to each point
photos = attachments[["note_uuid", "media"]].drop_duplicates("note_uuid")

dustbins = dustbins.merge(
    photos,
    left_on="uuid",
    right_on="note_uuid",
    how="left"
)

dustbins["photo_name"] = dustbins["media"].astype(str).str.split("/").str[-1]
dustbins.loc[dustbins["media"].isna(), "photo_name"] = ""

dustbins[["title", "note", "collector", "photo_name"]].head()


,title,note,collector,photo_name
0,Dustbin in front of tech hub,Not segregated,Aditi Saha,JPEG_20260902172440181.jpg
1,Dumping area,Too much garbage,Aditi Saha,JPEG_20260902173133529.jpg
2,In front Malhar hostel,Not segregated,Aditi Saha,JPEG_20260902174326739.jpg
3,In between malhar and des hostel,Not segregated,Aditi Saha,JPEG_20260902174624390.jpg
4,Entrance Gate of des hostel,Not segregated,Aditi Saha,JPEG_20260902174804565.jpg


In [5]:
# 5. Classify segregation status from the field note
def get_segregation(note):
    text = str(note).lower().replace("-", " ")
    text = " ".join(text.split())
    if "dry" in text:
        return "Dry waste"
    if "wet" in text or "green" in text:
        return "Wet waste"
    if "not segreg" in text or "non segreg" in text:
        return "Not segregated"
    if "segreg" in text:
        return "Segregated"
    return "Other"


colors = {
    "Not segregated": "#e67e22",
    "Dry waste": "#1f78b4",
    "Wet waste": "#33a02c",
    "Segregated": "#6a3d9a",
    "Other": "#7f7f7f",
}

dustbins["segregation"] = dustbins["note"].map(get_segregation)
dustbins["color"] = dustbins["segregation"].map(colors)
print(dustbins["segregation"].value_counts())


segregation
Not segregated    24
Dry waste         11
Other              6
Wet waste          4
Segregated         2
Name: count, dtype: int64


In [6]:
# 6. Build popup text for each point
popups = []

for i, row in dustbins.iterrows():
    location = f"{row.geometry.y:.5f}, {row.geometry.x:.5f}"
    html = (
        "<div style='width:240px;font-size:13px;line-height:1.4'>"
        + "<b>" + str(row["title"]) + "</b><br>"
        + "<b>Location:</b> " + location + "<br>"
        + "<b>Collected by:</b> " + str(row["collector"]) + "<br>"
        + "<b>Note:</b> " + str(row["note"]) + "<br>"
        + "<b>Segregation:</b> " + str(row["segregation"]) + "<br>"
        + "<b>Recorded at:</b> " + str(row["timestamp"])
    )
    if row["photo_name"] != "":
        html = (
            html
            + "<br><img src='photos/" + row["photo_name"]
            + "' style='margin-top:8px;width:100%;border-radius:4px'>"
        )
    else:
        html = html + "<br><i>No photograph attached</i>"
    html = html + "</div>"
    popups.append(html)

dustbins["popup"] = popups


In [7]:
# 7. Calculate map centre
center = [
    dustbins.geometry.y.mean(),
    dustbins.geometry.x.mean()
]
print(center)


[np.float64(13.712146693577598), np.float64(79.59102334263247)]


In [8]:
# 8. Create the Folium map
m = folium.Map(
    location=center,
    zoom_start=16,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles="OpenStreetMap",
    name="Streets"
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite"
).add_to(m)


In [9]:
# 9. Add points coloured by segregation status
for status, color in colors.items():
    subset = dustbins[dustbins["segregation"] == status]
    if len(subset) == 0:
        continue

    group = folium.FeatureGroup(name=status + " (" + str(len(subset)) + ")")

    for i, row in subset.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=7,
            fill=True,
            fill_opacity=0.9,
            weight=1,
            color="white",
            fill_color=color,
            tooltip=str(row["title"]) + " — " + status,
            popup=folium.Popup(row["popup"], max_width=280)
        ).add_to(group)

    group.add_to(m)


In [10]:
# 10. Add title, legend, north arrow, layer control, and fit the map
title_html = """
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
z-index:9999;background:white;padding:8px 14px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);text-align:center;
font-family:Arial;font-size:14px">
<b>Locations of the recorded waste bins, IIT Tirupati</b><br>
<span style="font-size:12px">Collected by Aditi Saha · QField · Sep 2026</span>
</div>
"""

legend_rows = ""
for status, color in colors.items():
    if len(dustbins[dustbins["segregation"] == status]) == 0:
        continue
    legend_rows = (
        legend_rows
        + "<div style='margin:3px 0'>"
        + "<span style='display:inline-block;width:12px;height:12px;"
        + "border-radius:50%;background:" + color + ";"
        + "margin-right:6px'></span>"
        + status
        + "</div>"
    )

legend_html = (
    """
<div style="position:fixed;bottom:40px;left:12px;z-index:9999;
background:white;padding:8px 12px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);font-family:Arial;font-size:12px">
<b>Legend</b><br>
"""
    + legend_rows
    + "</div>"
)

north_html = """
<div style="position:fixed;top:70px;right:12px;z-index:9999;
background:white;padding:6px 8px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);text-align:center;
font-family:Arial;font-size:14px;font-weight:bold;line-height:1.1">
↑<br>N
</div>
"""

m.get_root().html.add_child(Element(title_html))
m.get_root().html.add_child(Element(legend_html))
m.get_root().html.add_child(Element(north_html))

folium.LayerControl().add_to(m)

bounds = [
    [dustbins.geometry.y.min(), dustbins.geometry.x.min()],
    [dustbins.geometry.y.max(), dustbins.geometry.x.max()]
]
m.fit_bounds(bounds, padding=(30, 30))

m


In [11]:
# 11. Save the map as HTML for the website
Path("web").mkdir(exist_ok=True)
m.save("web/waste-bins.html")
print("Saved web/waste-bins.html")
print(len(dustbins), "waste-bin observations")


Saved web/waste-bins.html
47 waste-bin observations
